# Vertical Jump Coach Agent: Final Submission
WatSPEED Agentic AI · Course Project · Sean Tang  
Code: [github.com/seantanger/thp-jump-coach-agent](https://github.com/seantanger/thp-jump-coach-agent) (branch `main`) · Prompts: [`jump-coach-agent/prompts/`](https://github.com/seantanger/thp-jump-coach-agent/tree/main/jump-coach-agent/prompts) (`coach.v6.md`, `planner.v6.md`, `verifier.v4.md`, plus the shared rules in `AGENTS.md`)

**What this is.** A chat agent for my own vertical jump training, built on the John Evans / THP Strength YouTube transcripts. It is one chat agent with one sub-agent and an LLM judge built using LangGraph. The chat agent (the Coach) answers from the transcript library with cited sources, and it knows my profile, my saved plan, my training log and the last few turns. When I ask for a plan change, it calls the Planner, a second LangGraph agent, as a tool. The Planner writes the change, code checks it (schema, sources, calendar, spacing, equipment, per-session limits), a separate Verifier LLM judges it against the evidence, and nothing is saved until I approve it. The agents use the Claude API, and the project was built with the help of Claude Code.

**How I am using it.** This is not a toy plan. I started the program on Nov 9 and I am finishing week 2 this week (as of Nov 19). Week 3 is not written yet, so the last demo writes it. The log and plan below are my real training, including some slight knee pain after jumping last Saturday.

**Safe to run.** The demos run on a temporary copy of my data, so the notebook never changes my real plan or log.

In [1]:
import os, sys, shutil, tempfile
from pathlib import Path

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "jump-coach-agent"
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault("RETRIEVAL_MODE", "dense")  # dense (vector) search needs Ollama running

from src.kb.paths import data_dir
real = data_dir()

def copy_data():
    # a temporary copy of my data: big read-only data is linked, the athlete's files are copied
    copy = Path(tempfile.mkdtemp(prefix="jump_demo_"))
    for p in real.iterdir():
        if p.name == "athlete":
            shutil.copytree(p, copy / "athlete", ignore=shutil.ignore_patterns("checkpoints.sqlite*"))
        elif p.name != "traces":
            (copy / p.name).symlink_to(p)
    return copy

demo = copy_data()
os.environ["THP_DATA_DIR"] = str(demo)
print("demo data copy:", demo)

demo data copy: /var/folders/hh/qxwckjfd1v55gr1dlr4nngwh0000gn/T/jump_demo_j6wsm4bn


## 1. Architecture and orchestration
Three LLM roles with separate prompts, plus code checks. The Coach is the only thing the athlete talks to. The Planner and the Verifier are called by code and by the Coach's tools, never directly by me.

![Architecture](architecture.png)

| Role | Type | What it does |
|---|---|---|
| **Coach** | LLM agent (LangGraph: `coach` ⇄ `tools`) | Answers questions with cited sources, reads and writes the training log, decides when to call the Planner |
| **Planner** | LLM agent (LangGraph: `context` → `planner` ⇄ `tools` → `review` → `approve` → `save`) | Drafts new or changed plan weeks from reviewed templates |
| **Verifier** | LLM judge, separate prompt | Judges an answer or a plan draft against the cited evidence |
| **Code checks** | Plain code | Schema, citations are real, calendar and spacing, equipment, session limits, caps on tool calls |
| **Approval** | Human in the loop | The Planner pauses (LangGraph `interrupt`); only my approve click saves a new plan version |

## 2. Memory: what the agent knows about me
Memory is in files, not in the model. Each question starts with the date, my profile, my saved plan and my log, then the last 4 turns of the chat. The Coach can read further back through a tool.

| Memory | Where | Used for |
|---|---|---|
| Profile | `athlete/profile.json` | maxes, equipment, available days, knee status |
| Plan versions | `athlete/program/vNNNN.json` (never overwritten) | the saved plan, with a summary of each change |
| Training log | `athlete/log.jsonl` (append only) | what I did and how it felt |
| Chat history | last 4 turns, in the prompt | follow-up questions |
| Paused drafts | SQLite checkpointer | a draft waiting for my approval survives a restart |

In [2]:
import pandas as pd
from src.planner import store
from src.planner.log import load_log
from src.planner.render import outline_rows

profile, program = store.load_profile(), store.load_program()
print("Lifts (lb):", profile.lifts, "| knee:", profile.knee_pain.status, "-", profile.knee_pain.notes)
print(f"Saved plan: v{program.version}, {len(program.weeks)} weeks written, based on v{program.based_on}\n")
print("Plan versions on disk:", store.latest_version(), "| each change summary is kept in the file")
display(pd.DataFrame(outline_rows(program)))
print("\nTraining log:")
display(pd.DataFrame(load_log())[["date", "text", "knee_pain"]])

Lifts (lb): {'back_squat_lb': 225.0, 'half_squat_lb': 250.0, 'power_clean_lb': 185.0} | knee: ache - Knee pain in the past, now manageable.
Saved plan: v6, 2 weeks written, based on v5

Plan versions on disk: 6 | each change summary is kept in the file


,Month,Dates,Emphasis,Goal
0,1,Mon Nov 9 – Sun Dec 6,volume,Build work capacity with general work that loo...
1,2,Mon Dec 7 – Sun Jan 3,intensification (power and 6–8 rep strength),Intensify the month-1 volume with clean pulls ...
2,3,Mon Jan 4 – Sun Jan 31,absolute strength,Heavy squats and heavy clean pulls; eccentric ...
3,4,Mon Feb 1 – Sun Feb 28,"strength-speed (fast bars, eccentric overload)",Power clean to maxes; accentuated eccentric sq...
4,5,Mon Mar 1 – Sun Mar 28,speed-strength (bar speed),"Move bars fast (lighter cleans, power snatch, ..."
5,6,Mon Mar 29 – Sun Apr 25,specificity,"Specificity at its highest, with the most plyo..."



Training log:


,date,text,knee_pain
0,2026-11-09,"Week 1 lift day: power clean 135 lb 5x4, back ...",None
1,2026-11-11,"Power clean about 130-135 lb, back squat 4x10 ...",None
2,2026-11-13,Lying hamstring curl at 85 lb was too heavy; a...,None
3,2026-11-14,"Jump day: 10 penultimate-step cone reps, a slo...",None
4,2026-11-16,"Power clean 135 lb, back squat 4x9 at 160 lb, ...",None
5,2026-11-18,Back squat 4x9 at 160 lb and hip thrust 3x10.,None


**Memory demo.** One chat, three questions. The Coach has no search for these; it answers from the profile, the plan and the log. The second question depends on the first one's context, and the third is about how I felt.

In [3]:
import json
from src.coach.qa import answer_question

HISTORY = []
def tools_used(r):
    rows = [json.loads(l) for l in open(r["trace"])]
    return [(x["tool"], x["status"]) for x in rows if x.get("node") == "tools"]

def ask(question, fresh=False):
    """One chat turn. `fresh=True` starts a new conversation (no earlier turns)."""
    r = answer_question(question, athlete=True, history=None if fresh else HISTORY)
    if not fresh:
        HISTORY.append((question, r))
    print(f"YOU: {question}\n")
    print(r["compact"])
    verdict = r["check"].get("verifier", {}).get("verdict")
    print(f"\n[tools: {tools_used(r) or 'none'} | citation check passed: {r['check']['passed']} | Verifier: {verdict or 'n/a'} | {r['latency_s']:.0f}s]")
    return r

r = ask("What are my max lifts?")

YOU: What are my max lifts?

Your profile lists three maxes: back squat 225 lb, half squat about 250 lb (estimated) and power clean about 185 lb (estimated).

**For you:** Back squat: 225 lb, a deep squat. Half squat: about 250 lb, estimated. Power clean: about 185 lb, estimated, with 175 lb for 2 reps. This week's loads come from these numbers: power clean at 70% is about 130 lb, and back squat at 70% is about 158 lb. Your log shows you at 135 lb on power cleans and 160 lb on squats, so you're on target. If you've hit a new max, tell me the lift and weight and I can draft an update to your profile and plan.

[tools: none | citation check passed: True | Verifier: n/a | 5s]


In [4]:
r = ask("What part of the training cycle am I in right now?")

YOU: What part of the training cycle am I in right now?

You're in Month 1 of 6, the volume phase, and this is Week 2 of 4 (Nov 16–22).

**For you:** Month 1 runs Nov 9 – Dec 6. It builds work capacity with general work that looks little like jumping, and week 4 is an unload. Today is Thursday, a general day with the upper-body circuit and the core circuit. Month 2 starts Dec 7 and moves to intensification, with 6–8 rep strength, clean pulls and power cleans. The later months are absolute strength (Jan), strength-speed (Feb), speed-strength (Mar) and specificity (Mar 29 – Apr 25).

[tools: none | citation check passed: True | Verifier: n/a | 3s]


In [5]:
r = ask("How did I feel in the last few workouts?")

YOU: How did I feel in the last few workouts?

Your last few sessions felt manageable. The only things you flagged were a slight knee ache after jumping on Nov 14 and knee pain of 1-2/10 on Nov 16.

**For you:** Nov 14 (jump day): slight knee pain after about 15 max-effort jumps. Nov 16 (lift day): power clean 135 lb, back squat 4x9 at 160 lb and RDL 3x10 at 155 lb, with the knee at 1-2/10. Nov 18 (lift day): back squat 4x9 at 160 lb and hip thrust 3x10, with no feel or pain noted. Before that, on Nov 13, the lying hamstring curl at 85 lb was too heavy, so you dropped to about 70-75 lb. The knee ache is low and within the 2-3/10 limit in your jump-day plan. I'd keep logging it after jumps, especially this Saturday. If you want to log how Nov 18 felt, tell me.

[tools: none | citation check passed: True | Verifier: n/a | 5s]


**Memory over time.** The first message only holds the 10 most recent log entries. For longer spans the Coach calls `read_log`, which reads the whole log.

In [6]:
r = ask("How much have I improved since week 1?")

YOU: How much have I improved since week 1?

The log shows a small gain so far. Back squat went from 155 lb to 160 lb, and the other lifts are unchanged. Two weeks is too little data to call it real progress.

**For you:** Here is what I compared. Back squat: Nov 9 and Nov 11 were 4x10 at 155 lb. Nov 16 and Nov 18 were 4x9 at 160 lb, so +5 lb with one fewer rep per set. That matches the plan, which moves reps from 10 to 9 as load rises. Power clean: 135 lb on Nov 9 and Nov 16, and about 130-135 lb on Nov 11, so no change. This phase isn't aimed at clean maxes. RDL: about 155 lb on Nov 9 and 155 lb on Nov 16, so no change. Hip thrust: 270 lb then 320 lb on Nov 11 with no later weight logged, so I can't compare it. Hamstring curl: 85 lb was too heavy on Nov 13, and you aimed for 70-75 lb. Jumping: there are no jump heights in the log, only that you did about 15 max-effort jumps on Nov 14 with slight knee pain after. I can't say whether your vertical has changed. Your profile still lists 

## 3. Real questions, tools and citations
These are questions I actually asked while training. The Coach searches the transcript library (`search_library`, at most 4 searches per question), cites chunk IDs, and a code check confirms every cited chunk was really retrieved. The `For you` line connects the answer to my week; it is the Coach's own judgment, so it carries no citation.

In [7]:
r = ask("Can I do core work?")

YOU: Can I do core work?

Yes. Your plan already has a core circuit on Tuesdays and Thursdays, so today is a core day.

**For you:** Today, Thursday, do 3 sets of the core circuit after the upper-body circuit. Do a hollow hold for 20-30 s, a side plank for 20-30 s each side, and a hanging leg raise for 8-10 reps. Add the McGill Big 3 (curl-up, side bridge, bird dog) as 5-10 s holds. Keep it low-fatigue and don't push into back pain. The plan says not to add core on lift days (Mon, Wed, Fri). I'd keep to that so it doesn't tire your squats and cleans. If you'd like extra core work on another day, tell me and I can draft that change for you to review.

[tools: none | citation check passed: True | Verifier: n/a | 4s]


In [8]:
r = ask("Should I static stretch before jumping?")

YOU: Should I static stretch before jumping?

No. Keep stretching dynamic before jumping, because static stretching can lower power output. Save static holds for after the session, or use them only if tightness limits your positions.

- Skip static stretching before training. It can decrease force output a little. Use dynamic flexibility instead. [1]
- A dynamic warm-up could be bodyweight RDLs, side squats, leg swings and long lunges. Static flexibility has been shown in research to decrease power output. [2]
- If you want static stretching, do it after the warm-up and jumping. For example, hold a 2-minute ankle stretch with a plate on the knee if tight ankles make deep squatting hard. [3]

**For you:** Your Saturday jump session already starts with 3x45 s isometrics and low-effort warm-up jumps. I'd add a few minutes of dynamic flexibility, such as bodyweight squats, leg swings and long lunges, between the two. I'd also keep any static stretching for after the session.

**Sources**
1

## 4. Safeguards and guardrails
Safeguards are layered. Code does the checks that must not fail; LLMs do the judgment.

| Layer | Kind | What it stops |
|---|---|---|
| Citation check | code | an answer point with no source, or a source that was never retrieved |
| Verifier | LLM judge | a point the cited text does not actually support |
| Tool caps | code | runaway loops (4 searches, 1 plan call per question, 12 tool calls per draft) |
| Plan checks | code | invalid structure, missing sources, hard days too close together, equipment I don't have, loads or sets outside the template |
| Approval gate | human | any save: code writes a new version only after I approve |
| Prompt rules | LLM | style changes are accepted, safety or structure changes are declined; pain that sounds serious goes to a physio |
| Data rules | code | plan versions are never overwritten; log entries are append only |

These demos run on their own throwaway copy of my data, so a pretend injury report does not end up in the log used for week 3.

In [9]:
os.environ["THP_DATA_DIR"] = str(copy_data())

**A change the checks should refuse.** I ask for a jump session every day. The Planner drafts or declines, and code checks decide.

In [10]:
r = ask("Make every day a max-effort jump day this week.", fresh=True)

YOU: Make every day a max-effort jump day this week.

I didn't change your plan. Max-effort jumps every day aren't supported for you, so this week stays as written: Friday is a lift day and Saturday is your one max-effort jump session.

- The library suggests about one max-effort session a week, with 10–20 truly max-effort jumps. Two-foot jumpers can push that to roughly 20–30. [1][2]
- Intense jump sessions are like 'strapping dynamite to their legs'. Many jumpers need about 96 hours to recover from one. [3]
- Jumping every day was done by very young, light athletes with low verticals. That's not a fit for a 4-year lifter chasing a dunk. [3]

**For you:** You already have lifts Mon/Wed/Fri, and your knee had slight pain after last Saturday's jumps and was 1–2/10 on Monday. I'd keep Saturday as the one max-effort session and stop if knee pain goes above 2–3/10. If you want more jumping, I can draft a change to the Saturday session, such as more max-effort jumps within the 20–30 range.


**Serious pain.** The LLM decides this one (there is no keyword router any more). It should send me to a physio and not touch the plan.

In [11]:
r = ask("My knee popped on a landing and it is swollen and giving way.", fresh=True)

YOU: My knee popped on a landing and it is swollen and giving way.

A pop on landing with swelling and the knee giving way needs a physiotherapist or doctor now. I can't coach this, so stop all jumping, lifting and landing work until you've been assessed.

**For you:** I haven't changed your plan. Don't do Friday's lift or Saturday's jump session until you've been seen. I logged this on 2026-11-19, but the entry went in twice by mistake, so you may see a duplicate. Once you've been assessed, tell me what they say and I can draft a plan around it.

*See a physiotherapist or doctor promptly. A pop, swelling and giving way are signs of a possible structural knee injury. Until then, avoid loading the knee. Get urgent care if you can't bear weight, the knee looks deformed, or you get numbness or tingling in the leg.*

[tools: [('log_workout', 'success'), ('log_workout', 'success')] | citation check passed: True | Verifier: n/a | 7s]


**The Verifier is a real second LLM call.** To show it catches a bad claim, I take a real retrieved passage and a deliberately false point that cites it. A code check alone would pass this, because the cited chunk exists.

In [12]:
from src.verifier import verify_answer

real_r = answer_question("Should I static stretch before jumping?")
chunk_id = real_r["check"]["cited"][0]
evidence = {chunk_id: real_r["retrieved"][chunk_id]["text"]}
fake = {"summary": "Long static stretching before jumping adds 5 inches to your vertical.",
        "points": [{"text": "Hold static stretches for 3 minutes before jumping to add 5 inches to your vertical.", "cites": [chunk_id]}],
        "takeaway": None, "safety_note": None, "not_covered": False, "for_you": None}
verdict = verify_answer("Should I static stretch before jumping?", fake, evidence)
print("cited chunk exists:", chunk_id in real_r["retrieved"])
print("Verifier verdict:", verdict.verdict)
for item in verdict.items:
    print(f" - [{item.support}] {item.path}: {item.reason}")

cited chunk exists: True
Verifier verdict: fail
 - [unsupported] points[0]: Evidence says static stretching before training can decrease force output; no 3-minute hold or 5-inch vertical gain is stated.


In [13]:
os.environ["THP_DATA_DIR"] = str(demo)   # back to the main demo copy

## 5. Creating week 3
Week 3 starts Monday Nov 23, and I have not written it yet. I ask in plain words. The Coach calls `propose_plan_change`, which starts the Planner graph: it reads the template and my profile, plan and log, then writes the week; code checks it; the Verifier judges it; and the graph pauses for my approval. Nothing is saved yet.

In [14]:
r = ask("Please write week 3 of my plan.")
plan = r["plan"]
if plan is None or plan.status != "awaiting_approval":   # a draft can fail the checks; I ask once more and say what to keep
    print("\n--- first draft failed its checks, asking again ---\n")
    r = ask("Please try again. Keep week 3 the same as week 2 where you can and only change what the plan needs.")
    plan = r["plan"]
print("Planner status:", plan.status if plan else None, "| saved plan versions:", store.latest_version())

YOU: Please write week 3 of my plan.

A draft of week 3 (Mon Nov 23 to Sat Nov 28) is ready for you to review and approve. Nothing is saved until you approve it.

**For you:** The checks only partly supported the draft. The core lifts, accessories and the 35+5 min jump session are supported. The '3 sets' for the McGill core holds and the 'low-fatigue' label aren't backed by the evidence. The same goes for calling the cone drill 'two-foot'. Look at those items in the diff before you approve.

[tools: [('propose_plan_change', 'success'), ('search_library', 'success')] | citation check passed: True | Verifier: n/a | 138s]
Planner status: awaiting_approval | saved plan versions: 6


In [15]:
from src.planner.render import render_proposal, week_header, week_rows
from src.planner.schema import Week

print(render_proposal(plan))
for w in plan.proposal["replace_weeks"]:
    week = Week.model_validate_json(json.dumps(w))
    print("\n" + week_header(store.load_program(), week))
    display(pd.DataFrame(week_rows(week)))

What changes:
  Week of Mon Nov 23: 16 sessions
Why: Adds week 3 (2026-11-23 to 2026-11-28) of month 1 as a normal week. Lift days are Mon/Wed/Fri with power cleans at 5x4 (no percentage set), back squat at 4x8 at 70%, and one rotating accessory (hip thrust, lying hamstring curl, RDL). General days are Tue/Thu with a varied circuit plus McGill Big 3 core holds. Saturday is the jump session, with a penultimate-step cone drill before 35 min of low-effort jumps and 5 min of max-effort jumps.
⚠ weeks[2026-11-23]: partly supported — Core lifts, accessories and the 35+5 min jumps are supported. McGill '3 sets' and 'low-fatigue' are not in the evidence; 'two-foot' for the cone drill is unstated.

Month 1 of 6 · volume · Week 3 of 4


,Day,Type,Exercise,Sets×Reps,Load,Notes
0,Mon Nov 23,lift,power clean,5×4,,Power exercise first in the session; builds vo...
1,,,back squat,4×8,70%,About 158 lb (70% of your 225 lb squat).
2,,,hip thrust,3×10,,Hip-extension accessory; low risk.
3,Tue Nov 24,general,general circuit (varied movements),,,Many different movements for variety.
4,,,McGill Big 3 core holds,3 sets,,"Curl-up, side plank and bird dog, each as a 5-..."
5,Wed Nov 25,lift,power clean,5×4,,Power exercise first in the session; builds vo...
6,,,back squat,4×8,70%,About 158 lb (70% of your 225 lb squat).
7,,,lying hamstring curl,3×10,,Easy to add at the end; barely taxing.
8,Thu Nov 26,general,general circuit (varied movements),,,Many different movements for variety.
9,,,McGill Big 3 core holds,3 sets,,"Curl-up, side plank and bird dog, each as a 5-..."


**What the checks said.** Code failures, if any, and the Verifier's judgement of the draft against the cited evidence.

In [16]:
print("code check failures:", plan.failures or "none")
print("Verifier:", json.dumps(plan.verifier, indent=1)[:1500] if plan.verifier else "off")

code check failures: none
Verifier: {
 "warnings": [
  "weeks[2026-11-23]: partly supported \u2014 Core lifts, accessories and the 35+5 min jumps are supported. McGill '3 sets' and 'low-fatigue' are not in the evidence; 'two-foot' for the cone drill is unstated."
 ],
 "result": {
  "verdict": "pass",
  "items": [
   {
    "path": "weeks[2026-11-23]",
    "support": "partly",
    "reason": "Core lifts, accessories and the 35+5 min jumps are supported. McGill '3 sets' and 'low-fatigue' are not in the evidence; 'two-foot' for the cone drill is unstated."
   }
  ],
  "phase_rule_ok": true,
  "summary_matches_diff": true,
  "summary_issue": null,
  "addresses_request": true
 }
}


**Questions about the plan.** While the draft waits for my approval, I ask two things I have really asked while training. This is the same chat, so the Coach remembers the draft, and the saved plan is still unchanged.

In [17]:
r = ask("Is the power clean weight and rep scheme supposed to stay the same (135 lb, 5x4)?")

YOU: Is the power clean weight and rep scheme supposed to stay the same (135 lb, 5x4)?

Yes, for this week. Week 2 calls for power clean 5×4 @ 70%, about 130 lb, so your 135 lb is right on target.

**For you:** Your plan sets power clean at 5×4 @ 70% of your 185 lb max, which is about 130 lb. 135 lb is about 73%, and you logged 135 lb on Nov 9 and Nov 16, so the load and reps are holding steady. That fits Month 1, a volume phase that builds work capacity and unloads in week 4. The plan says heavier clean work comes in Month 2 (intensification), so I wouldn't expect the clean to climb in Month 1. I can't see the week 3 draft from here, so check the power clean line in the diff before you approve it. If you want it kept at 5×4 @ 70%, tell me and I can draft that change.

[tools: none | citation check passed: True | Verifier: n/a | 6s]


In [18]:
r = ask("Can I squat more than 160? I did 160 last week and it was pretty easy.")

YOU: Can I squat more than 160? I did 160 last week and it was pretty easy.

Yes, a small bump is reasonable. Volume weeks are meant to start easy, and you can add weight gradually if every set stays clean.

- In a volume phase, week one should feel pretty easy, around 3–5 reps in reserve. Then you build from there rather than going to failure on early sets. [1]
- Raise the percentage by no more than about 5% week to week. Sometimes 2.5% is enough. [2]
- Add 5–10 lb between sessions. If you miss reps, hold the weight until you hit them. If you drop well short, go lighter. [3]

**For you:** Your log shows 4x9 at 160 lb on Nov 16 and Nov 18, and the plan calls for about 158 lb (70%). I'd try about 165 lb on your next squat day (Friday). That is about a 3% rise. Keep the reps clean and the 3 s lowering, and stay under about 170 lb. This is Month 1, so I wouldn't push hard. Heavier squatting comes in Months 2 and 3. Your knee was 1–2/10 on Nov 16, so I'd back off if it climbs. If you want,

**Approval.** This is the human gate. Only now does code save a new version (here, in the demo copy).

In [19]:
from src.planner import run
done = run.resume(plan.thread_id, "approve")
print("status:", done.status, "| new version:", done.version, "| versions on disk:", store.latest_version())
print("\nChange summary kept with the version:\n", store.load_program().change_summary)

status: saved | new version: 7 | versions on disk: 7

Change summary kept with the version:
 Adds week 3 (2026-11-23 to 2026-11-28) of month 1 as a normal week. Lift days are Mon/Wed/Fri with power cleans at 5x4 (no percentage set), back squat at 4x8 at 70%, and one rotating accessory (hip thrust, lying hamstring curl, RDL). General days are Tue/Thu with a varied circuit plus McGill Big 3 core holds. Saturday is the jump session, with a penultimate-step cone drill before 35 min of low-effort jumps and 5 min of max-effort jumps.


Thank you Larry for teaching this course. I learned a lot and this was a great course for me to learn more about how AI agentic systems work under the hood. There is a lot of things that I was able to apply to this project, as well as projects at my work. I hope to expand on this project even more as I use it more for myself!